In [ ]:
import pyzed.sl as sl
import cv2
import numpy as np

INPUT_SVO  = 'data/driving_data_3.svo2'
OUTPUT_AVI = 'data/driving_data_3.avi'

zed = sl.Camera()

init_params = sl.InitParameters()
init_params.depth_mode = sl.DEPTH_MODE.NONE
init_params.set_from_svo_file(INPUT_SVO)

status = zed.open(init_params)
if status != sl.ERROR_CODE.SUCCESS:
    print('Camera Open: ' + repr(status) + '. Exit program.')
    zed.close()
    exit(1)

runtime   = sl.RuntimeParameters()
image_zed = sl.Mat()

width, height = 672, 376
fourcc     = cv2.VideoWriter_fourcc(*'XVID')
color_file = cv2.VideoWriter(OUTPUT_AVI, fourcc, 30, (width, height))

frame_count = 0
while True:
    err = zed.grab(runtime)
    if err == sl.ERROR_CODE.SUCCESS:
        zed.retrieve_image(image_zed, sl.VIEW.LEFT)
        color_bgr = image_zed.get_data()[:, :, :3]
        color_file.write(color_bgr)
        frame_count += 1
    elif err == sl.ERROR_CODE.END_OF_SVOFILE_REACHED:
        break

color_file.release()
zed.close()
print(f'Done — {frame_count} frames written to "{OUTPUT_AVI}"')